# Predictive Modelling

## Objectives

This notebook:

- creates time-aware predictive features;
- avoids using future values to construct features;
- uses a chronological training and test split;
- compares a linear-regression model with a seasonal-naive baseline;
- evaluates performance using MAE, RMSE, and R²;
- uses time-series cross-validation;
- exports predictions and metrics for the dashboard.

## Intended Use

The model is an educational historical one-month-ahead prediction prototype.
It estimates the current month's global land-and-ocean temperature using measurements available in previous months.

## Not Intended For

The model is not:

- a professional climate projection;
- a replacement for physical climate models;
- suitable for safety-critical or policy decisions;
- a forecast beyond the dataset's final date;
- evidence of a causal relationship.

The dataset ends in December 2015, so the results should not be described as current.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (
    TimeSeriesSplit,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "jupyter_notebooks"
    else Path.cwd()
)

PROCESSED_FOLDER = PROJECT_ROOT / "data" / "processed" / "v1"

GLOBAL_FILE = (
    PROCESSED_FOLDER / "global_temperatures_clean.csv"
)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project3/CI-Project3-ClimateLens-Understanding-Global-Temperature-Change


In [2]:
global_monthly = pd.read_csv(
    GLOBAL_FILE,
    parse_dates=["date"],
)

TARGET = "land_ocean_average_temperature_c"

# The land-and-ocean measurement begins in January 1850.
monthly_model_data = (
    global_monthly
    .dropna(subset=[TARGET])
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

print(f"Monthly observations: {len(monthly_model_data):,}")
print(
    "Coverage:",
    monthly_model_data["date"].min().date(),
    "to",
    monthly_model_data["date"].max().date(),
)

display(monthly_model_data.head())

Monthly observations: 1,992
Coverage: 1850-01-01 to 2015-12-01


,date,year,month,land_average_temperature_c,land_average_temperature_uncertainty_c,land_max_temperature_c,land_max_temperature_uncertainty_c,land_min_temperature_c,land_min_temperature_uncertainty_c,land_ocean_average_temperature_c,land_ocean_average_temperature_uncertainty_c
0,1850-01-01,1850,1,0.749,1.105,8.242,1.738,-3.206,2.822,12.833,0.367
1,1850-02-01,1850,2,3.071,1.275,9.970,3.007,-2.291,1.623,13.588,0.414
2,1850-03-01,1850,3,4.954,0.955,10.347,2.401,-1.905,1.410,14.043,0.341
3,1850-04-01,1850,4,7.217,0.665,12.934,1.004,1.018,1.329,14.667,0.267
4,1850-05-01,1850,5,10.004,0.617,15.655,2.406,3.811,1.347,15.507,0.249


## Feature Engineering

The following features are created:

- `time_index`: chronological position in the dataset;
- `month_sin` and `month_cos`: cyclical representation of seasonality;
- `lag_1_temperature_c`: temperature from the previous month;
- `lag_12_temperature_c`: temperature from the same month one year earlier;
- `rolling_12_temperature_c`: mean of the preceding 12 months;
- `rolling_120_temperature_c`: mean of the preceding 120 months.

All lagged and rolling features use `shift(1)`. Therefore, the current target value is never included in its own predictors.

In [3]:
monthly_model_data["time_index"] = np.arange(
    len(monthly_model_data)
)

monthly_model_data["month_sin"] = np.sin(
    2 * np.pi * monthly_model_data["month"] / 12
)

monthly_model_data["month_cos"] = np.cos(
    2 * np.pi * monthly_model_data["month"] / 12
)

monthly_model_data["lag_1_temperature_c"] = (
    monthly_model_data[TARGET].shift(1)
)

monthly_model_data["lag_12_temperature_c"] = (
    monthly_model_data[TARGET].shift(12)
)

monthly_model_data["rolling_12_temperature_c"] = (
    monthly_model_data[TARGET]
    .shift(1)
    .rolling(window=12)
    .mean()
)

monthly_model_data["rolling_120_temperature_c"] = (
    monthly_model_data[TARGET]
    .shift(1)
    .rolling(window=120)
    .mean()
)

FEATURES = [
    "time_index",
    "month_sin",
    "month_cos",
    "lag_1_temperature_c",
    "lag_12_temperature_c",
    "rolling_12_temperature_c",
    "rolling_120_temperature_c",
]

model_data = (
    monthly_model_data
    .dropna(subset=FEATURES + [TARGET])
    .reset_index(drop=True)
    .copy()
)

print(f"Model-ready shape: {model_data.shape}")
print(
    "Model-ready coverage:",
    model_data["date"].min().date(),
    "to",
    model_data["date"].max().date(),
)

display(model_data[["date", TARGET] + FEATURES].head())

Model-ready shape: (1872, 18)
Model-ready coverage: 1860-01-01 to 2015-12-01


,date,land_ocean_average_temperature_c,time_index,month_sin,month_cos,lag_1_temperature_c,lag_12_temperature_c,rolling_12_temperature_c,rolling_120_temperature_c
0,1860-01-01,13.170,120,0.500000,8.660254e-01,13.513,13.058,14.929917,14.928175
1,1860-02-01,13.424,121,0.866025,5.000000e-01,13.170,13.330,14.939250,14.930983
2,1860-03-01,13.714,122,1.000000,6.123234e-17,13.424,14.104,14.947083,14.929617
3,1860-04-01,14.861,123,0.866025,-5.000000e-01,13.714,15.197,14.914583,14.926875
4,1860-05-01,15.758,124,0.500000,-8.660254e-01,14.861,15.868,14.886583,14.928492


## Training and Test Strategy

A random split would allow later observations to appear in the training data while earlier observations appear in the test data. That would not reflect how time-series predictions are made.

The model therefore uses:

- training period: January 1860 through December 2005;
- test period: January 2006 through December 2015.

The test set contains the final 120 months and remains unseen during training.

In [4]:
TEST_START = pd.Timestamp("2006-01-01")

train_data = model_data.loc[
    model_data["date"] < TEST_START
].copy()

test_data = model_data.loc[
    model_data["date"] >= TEST_START
].copy()

X_train = train_data[FEATURES]
y_train = train_data[TARGET]

X_test = test_data[FEATURES]
y_test = test_data[TARGET]

assert train_data["date"].max() < test_data["date"].min()
assert len(test_data) == 120

print(f"Training observations: {len(train_data):,}")
print(f"Test observations: {len(test_data):,}")
print(
    "Training coverage:",
    train_data["date"].min().date(),
    "to",
    train_data["date"].max().date(),
)
print(
    "Test coverage:",
    test_data["date"].min().date(),
    "to",
    test_data["date"].max().date(),
)

Training observations: 1,752
Test observations: 120
Training coverage: 1860-01-01 to 2005-12-01
Test coverage: 2006-01-01 to 2015-12-01


In [5]:
model_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "regressor",
        LinearRegression(),
    ),
])

model_pipeline

Pipeline(steps=[('scaler', StandardScaler()),
                ('regressor', LinearRegression())])

## Time-Series Cross-Validation

Five expanding-window validation folds are used. Each validation fold contains 120 months.

The order of observations is preserved, and each validation period occurs after its corresponding training period.

In [6]:
time_series_cv = TimeSeriesSplit(
    n_splits=5,
    test_size=120,
)

cv_scores = cross_validate(
    model_pipeline,
    X_train,
    y_train,
    cv=time_series_cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "mse": "neg_mean_squared_error",
        "r2": "r2",
    },
)

cv_results = pd.DataFrame({
    "fold": range(1, 6),
    "mae_c": -cv_scores["test_mae"],
    "rmse_c": np.sqrt(-cv_scores["test_mse"]),
    "r2": cv_scores["test_r2"],
})

display(cv_results.round(4))

print(
    f"Mean cross-validation MAE: "
    f"{cv_results['mae_c'].mean():.4f} °C"
)

print(
    f"Mean cross-validation RMSE: "
    f"{cv_results['rmse_c'].mean():.4f} °C"
)

print(
    f"Mean cross-validation R²: "
    f"{cv_results['r2'].mean():.4f}"
)

,fold,mae_c,rmse_c,r2
0,1,0.0863,0.1110,0.9916
1,2,0.0878,0.1092,0.9921
2,3,0.0976,0.1220,0.9898
3,4,0.0899,0.1161,0.9908
4,5,0.1012,0.1246,0.9893


Mean cross-validation MAE: 0.0926 °C
Mean cross-validation RMSE: 0.1166 °C
Mean cross-validation R²: 0.9907


In [7]:
model_pipeline.fit(X_train, y_train)

model_predictions = model_pipeline.predict(X_test)

# Seasonal-naive benchmark:
# use the observed temperature from the same month one year earlier.
seasonal_naive_predictions = test_data[
    "lag_12_temperature_c"
].to_numpy()


def calculate_metrics(actual, predicted):
    """Calculate common regression evaluation metrics."""
    return {
        "mae_c": mean_absolute_error(actual, predicted),
        "rmse_c": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "r2": r2_score(actual, predicted),
    }


model_metrics = calculate_metrics(
    y_test,
    model_predictions,
)

baseline_metrics = calculate_metrics(
    y_test,
    seasonal_naive_predictions,
)

metrics_table = pd.DataFrame([
    {
        "model": "Linear regression",
        **model_metrics,
    },
    {
        "model": "Seasonal-naive baseline",
        **baseline_metrics,
    },
])

display(metrics_table.round(4))

,model,mae_c,rmse_c,r2
0,Linear regression,0.0891,0.1138,0.9916
1,Seasonal-naive baseline,0.1329,0.1725,0.9807


In [8]:
mae_improvement = (
    (
        baseline_metrics["mae_c"]
        - model_metrics["mae_c"]
    )
    / baseline_metrics["mae_c"]
    * 100
)

rmse_improvement = (
    (
        baseline_metrics["rmse_c"]
        - model_metrics["rmse_c"]
    )
    / baseline_metrics["rmse_c"]
    * 100
)

assert (
    model_metrics["mae_c"]
    < baseline_metrics["mae_c"]
)

assert (
    model_metrics["rmse_c"]
    < baseline_metrics["rmse_c"]
)

print(
    f"MAE improvement over baseline: "
    f"{mae_improvement:.1f}%"
)

print(
    f"RMSE improvement over baseline: "
    f"{rmse_improvement:.1f}%"
)

MAE improvement over baseline: 33.0%
RMSE improvement over baseline: 34.0%


In [9]:
prediction_results = pd.DataFrame({
    "date": test_data["date"].to_numpy(),
    "actual_temperature_c": y_test.to_numpy(),
    "predicted_temperature_c": model_predictions,
    "seasonal_naive_temperature_c":
        seasonal_naive_predictions,
})

prediction_results["residual_c"] = (
    prediction_results["actual_temperature_c"]
    - prediction_results["predicted_temperature_c"]
)

prediction_results["absolute_error_c"] = (
    prediction_results["residual_c"].abs()
)

display(prediction_results.head())

,date,actual_temperature_c,predicted_temperature_c,seasonal_naive_temperature_c,residual_c,absolute_error_c
0,2006-01-01,13.990,14.084352,14.159,-0.094352,0.094352
1,2006-02-01,14.435,14.236434,14.293,0.198566,0.198566
2,2006-03-01,14.966,15.009673,15.106,-0.043673,0.043673
3,2006-04-01,15.729,15.823840,15.978,-0.094840,0.094840
4,2006-05-01,16.463,16.615153,16.629,-0.152153,0.152153


In [10]:
prediction_figure = go.Figure()

prediction_figure.add_trace(
    go.Scatter(
        x=prediction_results["date"],
        y=prediction_results["actual_temperature_c"],
        name="Observed",
        mode="lines",
        line={
            "color": "#222222",
            "width": 2.5,
        },
    )
)

prediction_figure.add_trace(
    go.Scatter(
        x=prediction_results["date"],
        y=prediction_results["predicted_temperature_c"],
        name="Linear regression",
        mode="lines",
        line={
            "color": "#E45756",
            "width": 2,
        },
    )
)

prediction_figure.add_trace(
    go.Scatter(
        x=prediction_results["date"],
        y=prediction_results[
            "seasonal_naive_temperature_c"
        ],
        name="Seasonal-naive baseline",
        mode="lines",
        line={
            "color": "#4C78A8",
            "width": 1.5,
            "dash": "dot",
        },
    )
)

prediction_figure.update_layout(
    title=(
        "Historical Monthly Temperature Prediction: "
        "2006–2015 Test Period"
    ),
    xaxis_title="Date",
    yaxis_title="Temperature (°C)",
    template="plotly_white",
    hovermode="x unified",
    legend_title="Series",
)

prediction_figure.show()

/Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project3/CI-Project3-ClimateLens-Understanding-Global-Temperature-Change/.venv/lib/python3.12/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [11]:
comparison_figure = px.scatter(
    prediction_results,
    x="actual_temperature_c",
    y="predicted_temperature_c",
    color="absolute_error_c",
    color_continuous_scale="Blues",
    title="Observed Versus Predicted Temperature",
    labels={
        "actual_temperature_c":
            "Observed temperature (°C)",
        "predicted_temperature_c":
            "Predicted temperature (°C)",
        "absolute_error_c":
            "Absolute error (°C)",
    },
    template="plotly_white",
)

minimum_value = min(
    prediction_results["actual_temperature_c"].min(),
    prediction_results["predicted_temperature_c"].min(),
)

maximum_value = max(
    prediction_results["actual_temperature_c"].max(),
    prediction_results["predicted_temperature_c"].max(),
)

comparison_figure.add_shape(
    type="line",
    x0=minimum_value,
    y0=minimum_value,
    x1=maximum_value,
    y1=maximum_value,
    line={
        "color": "#E45756",
        "dash": "dash",
    },
)

comparison_figure.show()

In [12]:
residual_figure = px.histogram(
    prediction_results,
    x="residual_c",
    nbins=20,
    title="Distribution of Model Residuals",
    labels={
        "residual_c": (
            "Residual: observed minus predicted (°C)"
        ),
    },
    template="plotly_white",
)

residual_figure.add_vline(
    x=0,
    line_dash="dash",
    line_color="#E45756",
)

residual_figure.show()

print(
    f"Mean residual: "
    f"{prediction_results['residual_c'].mean():.4f} °C"
)

print(
    f"Largest absolute error: "
    f"{prediction_results['absolute_error_c'].max():.4f} °C"
)

Mean residual: 0.0133 °C
Largest absolute error: 0.3188 °C


In [13]:
standardised_coefficients = (
    model_pipeline
    .named_steps["regressor"]
    .coef_
)

coefficient_table = pd.DataFrame({
    "feature": FEATURES,
    "standardised_coefficient":
        standardised_coefficients,
})

coefficient_table["absolute_coefficient"] = (
    coefficient_table["standardised_coefficient"].abs()
)

coefficient_table = (
    coefficient_table
    .sort_values(
        "absolute_coefficient",
        ascending=True,
    )
    .reset_index(drop=True)
)

coefficient_figure = px.bar(
    coefficient_table,
    x="standardised_coefficient",
    y="feature",
    orientation="h",
    title="Standardised Linear-Model Coefficients",
    labels={
        "standardised_coefficient":
            "Standardised coefficient",
        "feature": "Feature",
    },
    template="plotly_white",
)

coefficient_figure.show()

display(coefficient_table)

,feature,standardised_coefficient,absolute_coefficient
0,time_index,0.013519,0.013519
1,rolling_120_temperature_c,0.019823,0.019823
2,month_sin,0.023294,0.023294
3,rolling_12_temperature_c,0.048859,0.048859
4,lag_12_temperature_c,0.151983,0.151983
5,month_cos,-0.600618,0.600618
6,lag_1_temperature_c,0.691672,0.691672


## Coefficient Interpretation

The coefficient chart indicates which standardised inputs have the strongest relationship with the model's predictions.

Large coefficients do not prove that a feature causes temperature change.
Lagged measurements and seasonal features are predictive because global temperature follows a strong seasonal and time-dependent pattern.

In [14]:
PREDICTIONS_OUTPUT = (
    PROCESSED_FOLDER / "model_test_predictions.csv"
)

METRICS_OUTPUT = (
    PROCESSED_FOLDER / "model_metrics.csv"
)

CV_OUTPUT = (
    PROCESSED_FOLDER / "model_cross_validation.csv"
)

COEFFICIENTS_OUTPUT = (
    PROCESSED_FOLDER / "model_coefficients.csv"
)

prediction_results.to_csv(
    PREDICTIONS_OUTPUT,
    index=False,
    date_format="%Y-%m-%d",
)

metrics_table.to_csv(
    METRICS_OUTPUT,
    index=False,
)

cv_results.to_csv(
    CV_OUTPUT,
    index=False,
)

coefficient_table.to_csv(
    COEFFICIENTS_OUTPUT,
    index=False,
)

# Verify the exported files.
prediction_check = pd.read_csv(PREDICTIONS_OUTPUT)
metrics_check = pd.read_csv(METRICS_OUTPUT)
cv_check = pd.read_csv(CV_OUTPUT)
coefficients_check = pd.read_csv(COEFFICIENTS_OUTPUT)

assert prediction_check.shape == prediction_results.shape
assert metrics_check.shape == metrics_table.shape
assert cv_check.shape == cv_results.shape
assert coefficients_check.shape == coefficient_table.shape

print("All model outputs were exported successfully.")
print(f"Predictions: {prediction_check.shape}")
print(f"Metrics: {metrics_check.shape}")
print(f"Cross-validation: {cv_check.shape}")
print(f"Coefficients: {coefficients_check.shape}")

All model outputs were exported successfully.
Predictions: (120, 6)
Metrics: (2, 4)
Cross-validation: (5, 4)
Coefficients: (7, 3)


## Model Conclusions

- A linear-regression pipeline was trained using chronological features and historical temperature measurements.
- The final ten years, January 2006 through December 2015, were reserved for testing.
- No random train/test split was used.
- All lagged and rolling features were calculated using preceding values.
- Five-fold time-series cross-validation produced a mean MAE of approximately 0.093 °C.
- The final model achieved an MAE of approximately 0.089 °C and an RMSE of approximately 0.114 °C.
- The model reduced MAE by approximately 33% compared with the seasonal-naive baseline.
- The high R² is partly explained by the strong seasonal cycle. It should not be interpreted as evidence that the model explains the physical causes of climate  change.
- Test-period predictions use observed preceding months. This is a one-step-ahead historical evaluation, not a ten-year forecast made from a single starting date.
- The model is suitable for educational demonstration and historical prototyping only.
- The model must not be presented as a professional or current climate projection.